# Multi-Agent Orchestration: Building Accessible AI Systems

Learn how to implement a multi-agent orchestration pattern where a central orchestrator agent coordinates multiple specialist agents to improve accessibility and user experience.

## Overview

This notebook demonstrates an advanced architectural pattern based on research into accessibility-focused AI systems. Instead of a single monolithic agent, we build:

1. **Orchestrator Agent**: Strategic manager that understands the task and delegates to specialists
2. **Summarization Agent**: Breaks down complex documents into digestible insights
3. **Settings Agent**: Handles user preferences and interface customization
4. **Shared Context**: All agents maintain awareness of the user's needs and document context

This pattern enables more intuitive interactions—users don't need to find the "right button"; the orchestrator routes their request to the appropriate specialist.

### Key Benefits

- **Modular Design**: Each agent has a specific expertise
- **Improved Accessibility**: Specialized handling of different user needs
- **Context Preservation**: Shared state across agent interactions
- **Scalability**: Easy to add new specialist agents
- **Better UX**: Intuitive, conversational interfaces

## Setup

### Prerequisites

- Python 3.8 or higher
- Anthropic API key from [console.anthropic.com](https://console.anthropic.com/)
- `anthropic` Python package (0.71.0 or later)

### Install Dependencies

```bash
pip install anthropic
```

### Environment Setup

Create a `.env` file in the parent directory with your API key:

```
ANTHROPIC_API_KEY=sk-ant-api03-...
```

In [ ]:
import json
import os
from dataclasses import dataclass

from anthropic import Anthropic
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Please set it in your environment or .env file.")

# Initialize client
client = Anthropic(api_key=API_KEY)

print("✓ API key loaded successfully")

## Architecture: Multi-Agent Orchestration

### System Design

```
User Input
    ↓
[Orchestrator Agent]
    ├─→ Route to Summarization Agent → Complex document analysis
    ├─→ Route to Settings Agent → User preferences
    └─→ Route to Context Manager → Shared state
    ↓
Unified Response
```

### Shared Context

All agents operate within a shared context that includes:
- The current document/task
- User accessibility preferences
- Conversation history
- Previous agent outputs

This context enables agents to be aware of previous interactions and provide coherent, coordinated responses.

In [ ]:
@dataclass
class UserContext:
    """Shared context across all agents."""

    user_id: str
    document_content: str
    accessibility_settings: dict
    conversation_history: list
    agent_outputs: dict

    def add_message(self, role: str, content: str):
        """Add a message to conversation history."""
        self.conversation_history.append({"role": role, "content": content})

    def save_agent_output(self, agent_name: str, output: str):
        """Save output from a specialist agent."""
        self.agent_outputs[agent_name] = output

    def get_context_summary(self) -> str:
        """Generate a summary of current context for agents."""
        return f"""
Current Context:
- User ID: {self.user_id}
- Document length: {len(self.document_content)} characters
- Accessibility settings: {json.dumps(self.accessibility_settings, indent=2)}
- Messages in history: {len(self.conversation_history)}
- Available agent outputs: {list(self.agent_outputs.keys())}
        """.strip()


# Initialize context for our demo
demo_context = UserContext(
    user_id="user_001",
    document_content="",
    accessibility_settings={
        "text_scale": 1.0,
        "high_contrast": False,
        "simplify_language": False,
        "audio_descriptions": True,
        "reading_speed": "normal",
    },
    conversation_history=[],
    agent_outputs={},
)

print("✓ UserContext initialized")
print(demo_context.get_context_summary())

## The Orchestrator Agent

The orchestrator is the strategic manager. It:
1. Analyzes user requests
2. Determines which specialist agent(s) should handle the task
3. Maintains the shared context
4. Coordinates responses from multiple agents
5. Provides the final unified response to the user

In [ ]:
class OrchestratorAgent:
    """Central coordinator that routes requests to specialist agents."""

    def __init__(self, client: Anthropic):
        self.client = client
        self.model = "claude-opus-4-6"
        self.specialist_agents = {}

    def register_specialist(self, name: str, agent):
        """Register a specialist agent."""
        self.specialist_agents[name] = agent

    def analyze_request(self, user_request: str, context: UserContext) -> dict:
        """Analyze request and determine specialist routing."""

        routing_prompt = f"""
You are an intelligent request router. Analyze the user's request and determine
which specialist agents should handle it.

Available specialist agents:
1. summarization_agent - Breaks down complex documents and extracts key insights
2. settings_agent - Handles user preferences and accessibility settings
3. context_agent - Retrieves and recalls previous information

Current context:
{context.get_context_summary()}

User request: {user_request}

Respond in JSON format with:
{{
  "primary_agent": "agent_name or null",
  "supporting_agents": ["agent_name", ...],
  "reasoning": "Brief explanation of routing decision",
  "special_instructions": "Any special instructions for the agents"
}}
        """

        response = self.client.messages.create(
            model=self.model, max_tokens=500, messages=[{"role": "user", "content": routing_prompt}]
        )

        try:
            # Extract JSON from response
            response_text = response.content[0].text
            # Find JSON in the response
            start = response_text.find("{")
            end = response_text.rfind("}") + 1
            if start >= 0 and end > start:
                routing_decision = json.loads(response_text[start:end])
            else:
                routing_decision = {
                    "primary_agent": None,
                    "supporting_agents": [],
                    "reasoning": "Unable to parse routing decision",
                    "special_instructions": "",
                }
        except json.JSONDecodeError:
            routing_decision = {
                "primary_agent": None,
                "supporting_agents": [],
                "reasoning": "Failed to parse routing decision",
                "special_instructions": "",
            }

        return routing_decision

    def orchestrate(self, user_request: str, context: UserContext) -> str:
        """Orchestrate the response by routing to specialists."""

        # Add user message to context
        context.add_message("user", user_request)

        # Analyze request
        routing_decision = self.analyze_request(user_request, context)

        print("\n🎯 Routing Decision:")
        print(f"  Primary Agent: {routing_decision.get('primary_agent', 'None')}")
        print(f"  Supporting: {routing_decision.get('supporting_agents', [])}")
        print(f"  Reasoning: {routing_decision.get('reasoning', '')}")

        # Collect responses from specialists
        specialist_responses = {}

        primary_agent = routing_decision.get("primary_agent")
        if primary_agent and primary_agent in self.specialist_agents:
            agent = self.specialist_agents[primary_agent]
            response = agent.process(
                user_request, context, routing_decision.get("special_instructions", "")
            )
            specialist_responses[primary_agent] = response
            context.save_agent_output(primary_agent, response)

        for supporting_agent_name in routing_decision.get("supporting_agents", []):
            if supporting_agent_name in self.specialist_agents:
                agent = self.specialist_agents[supporting_agent_name]
                response = agent.process(
                    user_request, context, routing_decision.get("special_instructions", "")
                )
                specialist_responses[supporting_agent_name] = response
                context.save_agent_output(supporting_agent_name, response)

        # Synthesize final response
        synthesis_prompt = f"""
You are the orchestrator synthesizing responses from specialist agents.
The user asked: {user_request}

Specialist responses:
{json.dumps(specialist_responses, indent=2)}

Provide a unified, coherent response that integrates all specialist insights.
Make it clear, accessible, and directly address the user's original request.
        """

        synthesis_response = self.client.messages.create(
            model=self.model,
            max_tokens=1000,
            messages=[{"role": "user", "content": synthesis_prompt}],
        )

        final_response = synthesis_response.content[0].text
        context.add_message("assistant", final_response)

        return final_response


orchestrator = OrchestratorAgent(client)
print("✓ Orchestrator Agent initialized")

## Specialist Agents

Each specialist agent handles a specific domain of expertise:

### 1. Summarization Agent
Breaks down complex documents into digestible insights, making information accessible.

### 2. Settings Agent
Dynamically adjusts user preferences like text scaling, language simplification, etc.

### 3. Context Agent
Manages and retrieves information from the shared context.

In [ ]:
class SummarizationAgent:
    """Specialist agent for document analysis and summarization."""

    def __init__(self, client: Anthropic):
        self.client = client
        self.model = "claude-sonnet-4-6"

    def process(self, request: str, context: UserContext, instructions: str = "") -> str:
        """Process request as summarization specialist."""

        prompt = f"""
You are a Summarization Specialist focused on making complex information accessible.

User request: {request}
Special instructions: {instructions or "None"}

Document to analyze (first 2000 chars):
{context.document_content[:2000]}

User accessibility preferences:
{json.dumps(context.accessibility_settings, indent=2)}

Your role:
1. Break down the document into key concepts
2. Extract the most important information
3. Adapt the explanation to the user's accessibility settings
4. Provide clear, step-by-step explanations

Provide your analysis in a structured format with:
- Key points (bulleted)
- Main concepts explained simply
- Recommendations for accessibility improvements
        """

        response = self.client.messages.create(
            model=self.model, max_tokens=800, messages=[{"role": "user", "content": prompt}]
        )

        return response.content[0].text


class SettingsAgent:
    """Specialist agent for user preferences and accessibility settings."""

    def __init__(self, client: Anthropic):
        self.client = client
        self.model = "claude-haiku-4-5"

    def process(self, request: str, context: UserContext, instructions: str = "") -> str:
        """Process request as settings specialist."""

        prompt = f"""
You are a Settings Specialist managing user accessibility preferences.

User request: {request}
Current accessibility settings:
{json.dumps(context.accessibility_settings, indent=2)}

Your role:
1. Understand what accessibility adjustments the user is requesting
2. Recommend appropriate settings changes
3. Explain how each setting affects the user experience
4. Suggest optimal configurations for common needs

Provide recommendations in JSON format for settings adjustments,
along with explanations of their impact.
        """

        response = self.client.messages.create(
            model=self.model, max_tokens=500, messages=[{"role": "user", "content": prompt}]
        )

        return response.content[0].text


class ContextAgent:
    """Specialist agent for context management and recall."""

    def __init__(self, client: Anthropic):
        self.client = client
        self.model = "claude-haiku-4-5"

    def process(self, request: str, context: UserContext, instructions: str = "") -> str:
        """Process request as context specialist."""

        # Retrieve relevant context
        recent_messages = context.conversation_history[-10:] if context.conversation_history else []

        prompt = f"""
You are a Context Specialist responsible for maintaining awareness of the conversation.

User request: {request}

Recent conversation history:
{json.dumps(recent_messages, indent=2)}

Previous agent outputs:
{json.dumps(context.agent_outputs, indent=2)}

Your role:
1. Identify relevant previous information
2. Note any context changes or updates
3. Highlight connections between current and previous requests
4. Suggest how to maintain continuity

Provide a brief analysis of the conversation context.
        """

        response = self.client.messages.create(
            model=self.model, max_tokens=500, messages=[{"role": "user", "content": prompt}]
        )

        return response.content[0].text


# Initialize specialist agents
summarization_agent = SummarizationAgent(client)
settings_agent = SettingsAgent(client)
context_agent = ContextAgent(client)

# Register specialists with orchestrator
orchestrator.register_specialist("summarization_agent", summarization_agent)
orchestrator.register_specialist("settings_agent", settings_agent)
orchestrator.register_specialist("context_agent", context_agent)

print("✓ Specialist agents initialized and registered")

## Example: Accessibility-Focused Document Analysis

Let's demonstrate the multi-agent orchestration with a realistic accessibility scenario.

### Scenario

A user with visual impairment is trying to understand a complex technical document. They want:
1. A simplified summary of the document
2. Audio descriptions of key sections
3. Recommendations for accessibility settings

In [ ]:
# Load sample document
sample_document = """
Web Accessibility Standards and Best Practices

1. Understanding WCAG 2.1

Web Content Accessibility Guidelines (WCAG) 2.1 is a set of recommendations for
making web content more accessible to people with disabilities. These guidelines
address accessibility for people with visual, auditory, physical, speech, cognitive,
language, learning, and neurological disabilities.

WCAG 2.1 is organized around four principles:
- Perceivable: Information must be presentable to users in ways they can perceive
- Operable: Interface components must be operable by users with various abilities
- Understandable: Text and operations must be clear and easy to understand
- Robust: Content must work across different technologies and assistive devices

2. Implementing Accessible Design

Accessible design is not a separate practice but rather an integral part of good
design. Key implementation strategies include:

- Use semantic HTML markup to provide structure and meaning
- Ensure sufficient color contrast (at least 4.5:1 for normal text)
- Provide alternative text for images describing their content and function
- Support keyboard navigation throughout the interface
- Test with real assistive technology users

3. Audio Descriptions and Multimodal Content

For users with visual impairments, audio descriptions provide narrated explanations
of visual content. Multimodal approaches that combine text, audio, and visual elements
benefit everyone, including users in noisy environments or those who prefer different
learning styles.
"""

# Update context with document
demo_context.document_content = sample_document
demo_context.accessibility_settings.update(
    {
        "text_scale": 1.5,
        "high_contrast": True,
        "simplify_language": True,
        "audio_descriptions": True,
        "reading_speed": "slow",
    }
)

print("✓ Sample document loaded")
print(f"✓ Document length: {len(sample_document)} characters")
print("✓ Accessibility settings configured")

### Example 1: Understanding the Document

The user asks for a summary suitable for their accessibility needs.

In [ ]:
print("\n" + "=" * 70)
print("USER REQUEST: Please summarize this accessibility document in simple terms")
print("=" * 70)

response = orchestrator.orchestrate(
    "Please summarize this accessibility document in simple terms that are easy to understand. "
    "Focus on the main concepts and how they apply to real users.",
    demo_context,
)

print("\n📤 ORCHESTRATOR RESPONSE:")
print("-" * 70)
print(response)
print("-" * 70)

### Example 2: Accessibility Settings

The user wants to optimize their accessibility settings for reading this type of technical content.

In [ ]:
print("\n" + "=" * 70)
print("USER REQUEST: What accessibility settings would help me read this better?")
print("=" * 70)

response = orchestrator.orchestrate(
    "What accessibility settings would help me read this technical content better? "
    "I find technical documentation challenging. Are there settings that could help?",
    demo_context,
)

print("\n📤 ORCHESTRATOR RESPONSE:")
print("-" * 70)
print(response)
print("-" * 70)

### Example 3: Context-Aware Follow-up

The user asks a follow-up question. Notice how the orchestrator maintains context from the previous interactions.

In [ ]:
print("\n" + "=" * 70)
print("USER REQUEST: Can you explain WCAG principles in more detail?")
print("=" * 70)

response = orchestrator.orchestrate(
    "Can you explain the WCAG principles mentioned in the document? "
    "I want to understand each one better for my own projects.",
    demo_context,
)

print("\n📤 ORCHESTRATOR RESPONSE:")
print("-" * 70)
print(response)
print("-" * 70)

## Conversation Analysis

Let's analyze how the multi-agent system handled the conversation.

In [ ]:
print("\n" + "=" * 70)
print("CONVERSATION ANALYSIS")
print("=" * 70)

print("\n📊 Conversation Statistics:")
print(f"  Total messages: {len(demo_context.conversation_history)}")
print(
    f"  User messages: {sum(1 for m in demo_context.conversation_history if m['role'] == 'user')}"
)
print(
    f"  Assistant messages: {sum(1 for m in demo_context.conversation_history if m['role'] == 'assistant')}"
)

print("\n🤖 Specialist Agents Used:")
for agent_name, output in demo_context.agent_outputs.items():
    print(f"  - {agent_name}: {len(output)} characters")

print("\n♿ Accessibility Settings Applied:")
for setting, value in demo_context.accessibility_settings.items():
    print(f"  - {setting}: {value}")

print("\n📝 Recent Message History:")
print("-" * 70)
for i, msg in enumerate(demo_context.conversation_history[-4:], 1):
    role = msg["role"].upper()
    content = msg["content"][:100] + "..." if len(msg["content"]) > 100 else msg["content"]
    print(f"{i}. [{role}] {content}")

## Key Benefits of Multi-Agent Orchestration

### 1. **Modular Expertise**
Each agent specializes in one area (summarization, settings, context), making them easier to develop and improve independently.

### 2. **Better Accessibility**
Specialist agents can be optimized for specific accessibility needs without compromising general functionality.

### 3. **Context Preservation**
Shared context across agents ensures consistency and allows agents to build on each other's work.

### 4. **Intelligent Routing**
The orchestrator routes requests to the most appropriate specialist(s), improving response quality and efficiency.

### 5. **Scalability**
New specialist agents can be added without modifying the orchestrator's core logic.

### 6. **Better UX**
Users interact naturally without needing to know which "button" to click. The system figures out what they need.

## Extensions and Customization

### Adding New Specialist Agents

To add a new specialist agent:

```python
class CustomAgent:
    def __init__(self, client: Anthropic):
        self.client = client
        self.model = "claude-sonnet-4-6"

    def process(self, request: str, context: UserContext, instructions: str = "") -> str:
        # Implement your agent logic
        prompt = f"Your specialized prompt here: {request}"
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

# Register with orchestrator
custom_agent = CustomAgent(client)
orchestrator.register_specialist("custom_agent", custom_agent)
```

### Real-World Applications

This pattern works well for:
- **StreetReaderAI**: Visual guide with visual description and question-answering agents
- **Multimodal Video Player**: Video parsing agent + interactive Q&A agent
- **Educational Platforms**: Content explanation + learning progress + assessment agents
- **Document Processing**: OCR agent + analysis agent + formatting agent

## Conclusion

Multi-agent orchestration represents a powerful approach to building more accessible, responsive AI systems. By separating concerns into specialist agents and coordinating them through a smart orchestrator, we can:

1. **Improve Accessibility**: Specialists can focus on specific accessibility needs
2. **Enhance Quality**: Each agent becomes expert in its domain
3. **Enable Scalability**: New capabilities can be added as new specialists
4. **Create Better UX**: Users interact intuitively without technical complexity

This architecture opens possibilities for AI systems that are not just powerful, but genuinely usable by everyone.

### Next Steps

1. Implement additional specialist agents for your use case
2. Optimize agent selection based on user behavior
3. Add persistent context storage for multi-session interactions
4. Implement feedback mechanisms to improve routing decisions
5. Monitor agent performance and cost efficiency